# 05 — Final held-out test evaluation

This notebook is a thin Colab entry point. Reusable code lives in `src/cross_image_glot`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

from pathlib import Path
import sys

REPO_URL = "https://github.com/TomerBurman/CrossImagePatchGraph.git"
REPO_DIR = Path("/content/CrossImagePatchGraph_repo")

if "<YOUR_GITHUB_USERNAME>" in REPO_URL:
    raise ValueError("Set REPO_URL to your GitHub repository before running this notebook.")

if not REPO_DIR.exists():
    !git clone "$REPO_URL" "$REPO_DIR"
else:
    !git -C "$REPO_DIR" pull

%cd /content/CrossImagePatchGraph_repo
!pip install -q -r requirements.txt

SRC_DIR = REPO_DIR / "src"

if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

In [ ]:
import json
import torch

from cross_image_glot.config import DEFAULT_PATHS

paths = DEFAULT_PATHS
paths.ensure_directories()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("Drive root:", paths.drive_root)
print("Local runtime root:", paths.local_root)

Do not use this notebook for model selection. Run it only after the architecture and hyperparameters are frozen.

In [ ]:
from cross_image_glot.storage import restore_feature_splits, atomic_json_save
from cross_image_glot.data import MiniImageNetFeatureDataset, FewShotFeatureEpisodeDataset
from cross_image_glot.baselines import evaluate_frozen_baseline
from cross_image_glot.graph_builder import ClassConditionedPatchGraphBuilder
from cross_image_glot.models import CrossImageGraphMatcher, BaselinePreservingResidualMatcher
from cross_image_glot.training import evaluate_episode_dataset, evaluate_residual_dataset

restore_feature_splits(["test"], paths.drive_feature_dir, paths.local_feature_dir)
test_features = MiniImageNetFeatureDataset(paths.local_feature_dir, "test", max_cached_shards=6)
test_episodes = FewShotFeatureEpisodeDataset(
    test_features, n_way=5, k_shot=5, queries_per_class=15,
    num_episodes=600, seed=20_000, vary_by_epoch=False,
)

In [ ]:
# Always report frozen baselines on exactly the same test episodes.
cls_metrics = evaluate_frozen_baseline(test_episodes, "cls", device, 600, temperature=0.1)
mean_metrics = evaluate_frozen_baseline(test_episodes, "mean_patch", device, 600, temperature=0.1)
print("CLS:", cls_metrics)
print("Mean patch:", mean_metrics)

In [ ]:
MODEL_KIND = "residual"  # "graphsage" or "residual"
EXPERIMENT_NAME = "residual_v1" if MODEL_KIND == "residual" else "graphsage_v1"
config = json.loads(Path(f"configs/{'residual_5shot' if MODEL_KIND == 'residual' else 'graphsage_5shot'}.json").read_text())
graph_builder = ClassConditionedPatchGraphBuilder(
    grid_size=tuple(test_features.metadata["grid_size"]), top_k=config["top_k"],
    min_similarity=None, graph_dtype=torch.float32, similarity_device=device,
)
graph_matcher = CrossImageGraphMatcher(
    input_dim=config["input_dim"], hidden_dim=config["hidden_dim"], num_layers=config["num_layers"],
    dropout=config["dropout"], temperature=config.get("graph_temperature", config.get("temperature", 0.1)),
    learnable_temperature=False,
)
if MODEL_KIND == "residual":
    model = BaselinePreservingResidualMatcher(graph_matcher, config["initial_residual_scale"])
else:
    model = graph_matcher
checkpoint = torch.load(paths.drive_checkpoint_dir / EXPERIMENT_NAME / "best.pt", map_location="cpu", weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
model.to(device)

In [ ]:
if MODEL_KIND == "residual":
    model_metrics = evaluate_residual_dataset(
        model, graph_builder, test_episodes, device, 600,
        config["graph_microbatch_size"], config["cls_temperature"],
        log_interval=20, split_name="test",
    )
else:
    model_metrics = evaluate_episode_dataset(
        model, graph_builder, test_episodes, device, 600,
        config["graph_microbatch_size"], log_interval=20, split_name="test",
    )
results = {
    "model_kind": MODEL_KIND,
    "model": model_metrics.to_dict(),
    "cls_baseline": cls_metrics.to_dict(),
    "mean_patch_baseline": mean_metrics.to_dict(),
}
output = paths.drive_results_dir / EXPERIMENT_NAME / "test_metrics.json"
atomic_json_save(results, output)
print(results)
print("Saved:", output)